In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_models, get_features, model_names, ModelTypes

from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [2]:
enabled_models = ('dv2', 'dvt', 'alibi_dv2_cb')

models = get_models(enabled_models, '../../trained_models', DEVICE)

In [3]:
image_fnames = ['diff_shapes_518.png', 'wuppertal.jpg', 'bubbles_.png', '000004.jpg']
imgs: list[Image.Image] = []
ks = [4, 4, 3, 3]
results: dict[ModelTypes, list[np.ndarray]] = {mt: [] for mt in enabled_models}

In [4]:
for img_fname in image_fnames:
    img = Image.open(f'data/PCA/{img_fname}').convert('RGB')
    img = img.resize((518, 518))
    imgs.append(img)

In [5]:
%%capture
for key, model in models.items():
    for i, img in enumerate(imgs):
        feats_li = []
        feats = get_features(model, img, True, False, device=DEVICE)
        h, w, c = feats.shape
        flat = feats.reshape((h * w, c))
        flat = scale(flat)
        reduced = KMeans(ks[i], n_init=10).fit_predict(flat)
        reduced_2D = reduced.reshape((h, w))
        results[key].append(reduced_2D)

In [6]:
%%capture
from skimage.color import label2rgb
from PIL.ImageColor import getcolor

COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]


W, H = 3, 3
FS = 30
add_custom_font('resources/fonts', 'Grotesk')

N_COLS = len(image_fnames)
N_ROWS = len(enabled_models) + 1

fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * W, N_ROWS * H))

for i, img in enumerate(imgs):
    ax = axs[0, i]
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])

    for j, key in enumerate(enabled_models):
        ax = axs[j + 1, i]
        selected_ch = results[key][i]
        remapped = label2rgb(selected_ch, colors=COLORS, bg_label=-1)
        ax.imshow(remapped, cmap='tab10')
        ax.set_xticks([])
        ax.set_yticks([])

        if i == 0:
            model_name = model_names[key]
            model_name = model_name.replace('ALiBi(CB)-Dv2', 'ALiBi-Dv2')
            weight = 700 if 'ALiBi' in model_name else 500
            axs[j + 1, 0].set_ylabel(model_name, fontsize=FS, weight=weight)
            # axs[i, 2].set_title('ALiBi-Dv2', fontsize=FS, weight=700)
plt.tight_layout()

plt.savefig('saved/04.png', dpi=300)